# Train My LLM on Google Colab

Select a GPU runtime, set the GitHub URL in the configuration cell, then run the cells in order. Checkpoints are saved to Google Drive and resume automatically. Use `MODE = "lora"` for GPU-friendly conversation tuning on top of a pretrained checkpoint.

In [ ]:
# Edit these values before running the remaining cells.
REPO_URL = "PASTE_YOUR_GITHUB_REPO_URL_HERE"
REPO_DIR = "/content/my-llm"
DRIVE_ROOT = "/content/drive/MyDrive/my-llm"
CHECKPOINT_DIR = f"{DRIVE_ROOT}/checkpoints"
DATA_DIR = f"{DRIVE_ROOT}/datasets"
MODE = "lora"             # "pretrain", "finetune", or "lora"
STEPS = 20                     # use 2000+ for conversation tuning
EPOCHS = 1
CHECKPOINT_EVERY = 100
DTYPE = "auto"                # CUDA uses BF16 when supported, otherwise FP16
LORA_RANK = 16
LORA_ALPHA = 32.0
LORA_DROPOUT = 0.05
GENERATE_PUBLIC_DOMAIN_DATA = False

assert REPO_URL != "PASTE_YOUR_GITHUB_REPO_URL_HERE", "Set REPO_URL first."


In [ ]:
import subprocess, sys
from urllib.parse import urlparse
from pathlib import Path

repo_slug = urlparse(REPO_URL).path.strip('/').removesuffix('.git')
requirements_url = f"https://raw.githubusercontent.com/{repo_slug}/main/requirements-colab.txt"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', requirements_url], check=True)

from google.colab import drive
drive.mount('/content/drive')
if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
print('Repository:', REPO_DIR)
print('Checkpoints:', CHECKPOINT_DIR)


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())
else:
    raise RuntimeError('Select Runtime > Change runtime type > GPU before training.')


## Dataset setup

Pretraining reads `training_data.txt`; full conversation tuning and LoRA tuning read `chat_dataset.jsonl`. The LoRA stage freezes the pretrained model and saves only adapter weights, so Drive checkpoints are much smaller. Conversation data should not be redistributed unless its license permits it.

In [ ]:
import shutil
repo_data = Path(REPO_DIR) / 'ai-model' / ('training_data.txt' if MODE == 'pretrain' else 'chat_dataset.jsonl')
drive_data = Path(DATA_DIR) / repo_data.name
DATA_PATH = drive_data if drive_data.exists() else repo_data

if GENERATE_PUBLIC_DOMAIN_DATA and MODE == 'pretrain':
    subprocess.run([sys.executable, 'build_dataset.py', '--target_mb', '500'], cwd=f'{REPO_DIR}/ai-model', check=True)
    shutil.copy2(f'{REPO_DIR}/ai-model/training_data.txt', f'{DATA_DIR}/training_data.txt')
    DATA_PATH = Path(DATA_DIR) / 'training_data.txt'

print('Training data:', DATA_PATH)
print('Size (MB):', DATA_PATH.stat().st_size / (1024 * 1024))
if DATA_PATH.stat().st_size < 20 * 1024 * 1024 and MODE == 'pretrain':
    print('WARNING: this is a small corpus for a 500M model.')


In [ ]:
# Re-running this cell resumes from the latest Drive checkpoint.
command = [sys.executable, 'colab_train.py', '--mode', MODE, '--data', str(DATA_PATH), '--checkpoint-dir', CHECKPOINT_DIR, '--steps', str(STEPS), '--epochs', str(EPOCHS), '--checkpoint-every', str(CHECKPOINT_EVERY), '--dtype', DTYPE]
if MODE in ('finetune', 'lora'):
    command += ['--pretrained', f'{CHECKPOINT_DIR}/pretrain_500m_latest.pth']
if MODE == 'lora':
    command += ['--lora-rank', str(LORA_RANK), '--lora-alpha', str(LORA_ALPHA), '--lora-dropout', str(LORA_DROPOUT)]
print('Running:', ' '.join(command))
subprocess.run(command, cwd=f'{REPO_DIR}/ai-model', check=True)


In [ ]:
for item in sorted(Path(CHECKPOINT_DIR).glob('*')):
    print(f'{item.name}: {item.stat().st_size / (1024 * 1024):.1f} MB')

if MODE == 'lora':
    print('\nTo run the chat app with the adapter, copy or mount both files and set:')
    print('LLM_BASE_CHECKPOINT=pretrain_500m_latest.pth')
    print('LLM_ADAPTER_CHECKPOINT=small_llm_lora.pth')
